# Network Degree Tables

Builds per-school degree tables for 2019 and 2023:
- **HS table**: one row per high school per year — in-degree = # unique colleges that visited
- **College table**: one row per college per year — out-degree = # unique HS visited

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## Load & Filter

In [2]:
df_raw = pd.read_csv('../data/final_data_v4.csv')

# Filter: high school grades only (12 = highest grade is 12th, 13 = includes PK/ungraded but goes through 12)
hs_mask = df_raw['hs_highest_grade_offered'].isin([12.0, 13.0])

# Filter: drop rows where college is missing on ALL key college variables
col_vars = ['col_name', 'col_inst_control', 'col_inst_size', 'col_endow_total',
            'col_endow_per_fte', 'col_number_applied', 'col_number_admitted',
            'col_number_enrolled_total', 'col_acceptance_rate', 'col_tuition_published']
missing_col_mask = df_raw[col_vars].isna().all(axis=1)

df = df_raw[hs_mask & ~missing_col_mask].copy()

print(f'Raw rows:      {len(df_raw):,}')
print(f'Filtered rows: {len(df):,}  ({len(df)/len(df_raw):.1%} retained)')
print(f'Colleges dropped (no data): {missing_col_mask.sum():,}  ({missing_col_mask.mean():.1%} of raw rows)')
print(f'Unique colleges: {df["college_id"].nunique()}')
print(f'Unique HS:       {df["hs_id"].nunique()}')
print()
for yr in [2019, 2023]:
    s = df[df['cycle'] == yr]
    print(f'{yr}: {len(s):,} rows | {s["college_id"].nunique():,} colleges | {s["hs_id"].nunique():,} HS')

Raw rows:      1,048,575
Filtered rows: 375,827  (35.8% retained)
Colleges dropped (no data): 27,588  (2.6% of raw rows)
Unique colleges: 2706
Unique HS:       26690

2019: 100,540 rows | 2,632 colleges | 21,692 HS
2023: 60,081 rows | 2,536 colleges | 18,495 HS


## Save Filtered Dataset

In [3]:
df.to_csv('../data/final_data_v4_filtered.csv', index=False)
print(f'Saved ../data/final_data_v4_filtered.csv  ({len(df):,} rows)')

Saved ../data/final_data_v4_filtered.csv  (375,827 rows)


## Build Degree Tables

For each year, count:
- **Out-degree** (college): number of *unique* high schools that college visited
- **In-degree** (HS): number of *unique* colleges that visited that high school

In [4]:
years = [2019, 2023]

# --- College out-degree table ---
# Deduplicate so each college-HS pair counts once per year, then count unique HS per college
college_degree_parts = []
for yr in years:
    sub = df[df['cycle'] == yr].copy()

    # unique HS visited per college
    out_deg = (
        sub.drop_duplicates(subset=['college_id', 'hs_id'])
           .groupby('college_id')
           .size()
           .reset_index(name='n_hs_visited')
    )

    # all college-level attributes (take first occurrence — stable within a year)
    col_attrs = (
        sub.drop_duplicates('college_id')
           [['college_id', 'col_name', 'col_city', 'col_st', 'col_zip',
             'col_ctyname', 'col_ctyfips', 'col_type',
             'col_inst_control', 'col_inst_size',
             'col_endow_total', 'col_endow_per_fte',
             'col_number_applied', 'col_number_admitted',
             'col_number_enrolled_total', 'col_enrollment_rate',
             'col_acceptance_rate', 'col_tuition_published', 'col_tuition_fees_ft']]
    )

    merged = out_deg.merge(col_attrs, on='college_id', how='left')
    merged.insert(1, 'year', yr)
    college_degree_parts.append(merged)

college_degree_df = pd.concat(college_degree_parts, ignore_index=True)

print('College out-degree table shape:', college_degree_df.shape)
print(college_degree_df.head(6).to_string(index=False))

College out-degree table shape: (5168, 21)
 college_id  year  n_hs_visited                            col_name   col_city col_st   col_zip col_ctyname  col_ctyfips  col_type  col_inst_control  col_inst_size  col_endow_total  col_endow_per_fte  col_number_applied  col_number_admitted  col_number_enrolled_total  col_enrollment_rate  col_acceptance_rate  col_tuition_published  col_tuition_fees_ft
     100654  2019            23            ALABAMA A & M UNIVERSITY     NORMAL     AL 35762.000     MADISON     1089.000     1.000             1.000          3.000              NaN                NaN            9579.000             8789.000                   1710.000                0.195                0.918              17220.000            18634.000
     100663  2019           235 UNIVERSITY OF ALABAMA AT BIRMINGHAM BIRMINGHAM     AL 35294.000   JEFFERSON     1073.000     1.000             1.000          5.000              NaN                NaN            8298.000             6112.000         

In [5]:
# --- HS in-degree table ---
hs_degree_parts = []
for yr in years:
    sub = df[df['cycle'] == yr].copy()

    # unique colleges visiting per HS
    in_deg = (
        sub.drop_duplicates(subset=['college_id', 'hs_id'])
           .groupby('hs_id')
           .size()
           .reset_index(name='n_colleges_visiting')
    )

    # all HS-level attributes (take first occurrence — stable within a year)
    hs_attrs = (
        sub.drop_duplicates('hs_id')
           [['hs_id', 'hs_name', 'hs_city', 'hs_state', 'hs_zip',
             'hs_ctyname', 'hs_cty_fips', 'hs_lat', 'hs_long',
             'school_type', 'hs_school_level', 'hs_highest_grade_offered',
             'hs_charter', 'hs_magnet', 'hs_title_i_status',
             'hs_urban_centric_locale', 'hs_enrollment', 'hs_students_per_teacher',
             'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic',
             'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more',
             'hs_pct_free_or_reduced_price_lunch']]
    )

    merged = in_deg.merge(hs_attrs, on='hs_id', how='left')
    merged.insert(1, 'year', yr)
    hs_degree_parts.append(merged)

hs_degree_df = pd.concat(hs_degree_parts, ignore_index=True)

print('HS in-degree table shape:', hs_degree_df.shape)
print(hs_degree_df.head(6).to_string(index=False))

HS in-degree table shape: (40187, 28)
   hs_id  year  n_colleges_visiting                                     hs_name        hs_city hs_state    hs_zip hs_ctyname  hs_cty_fips  hs_lat  hs_long school_type hs_school_level  hs_highest_grade_offered  hs_charter  hs_magnet  hs_title_i_status  hs_urban_centric_locale  hs_enrollment  hs_students_per_teacher  hs_pct_white  hs_pct_black  hs_pct_hispanic  hs_pct_asian  hs_pct_aian  hs_pct_nhpi  hs_pct_two_or_more  hs_pct_free_or_reduced_price_lunch
00000044  2019                   17                 HOLY SPIRIT CATHOLIC SCHOOL     TUSCALOOSA       AL 35405.000 TUSCALOOSA     1125.000  33.174  -87.529     private        combined                    12.000     -10.000    -10.000            -10.000                  -10.000        464.000                   10.791         0.651         0.106            0.078         0.028        0.002        0.000               0.034                             -10.000
00000237  2019                    1 HOLY FAMILY 

## Summary Statistics

In [6]:
print('=== College Out-Degree (# unique HS visited) ===')
college_summary = (
    college_degree_df.groupby('year')['n_hs_visited']
    .agg(
        n_colleges='count',
        mean='mean',
        median='median',
        std='std',
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p99=lambda x: x.quantile(0.99),
        max='max',
    )
    .round(1)
)
print(college_summary.to_string())

print()
print('=== HS In-Degree (# unique colleges visiting) ===')
hs_summary = (
    hs_degree_df.groupby('year')['n_colleges_visiting']
    .agg(
        n_hs='count',
        mean='mean',
        median='median',
        std='std',
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p99=lambda x: x.quantile(0.99),
        max='max',
    )
    .round(1)
)
print(hs_summary.to_string())

=== College Out-Degree (# unique HS visited) ===
      n_colleges   mean  median    std   p25    p75    p90     p99  max
year                                                                   
2019        2632 38.200  20.000 53.900 8.000 45.000 90.000 263.400  644
2023        2536 23.700  14.000 28.400 6.000 30.000 56.000 139.000  258

=== HS In-Degree (# unique colleges visiting) ===
       n_hs  mean  median   std   p25   p75    p90    p99  max
year                                                          
2019  21692 4.600   3.000 5.200 2.000 6.000 10.000 23.000  194
2023  18495 3.200   2.000 3.300 1.000 4.000  7.000 17.000   88


## Save Degree Tables

In [7]:
college_degree_df.to_csv('../data/college_out_degree.csv', index=False)
hs_degree_df.to_csv('../data/hs_in_degree.csv', index=False)

print(f'Saved ../data/college_out_degree.csv  ({len(college_degree_df):,} rows)')
print(f'Saved ../data/hs_in_degree.csv        ({len(hs_degree_df):,} rows)')
print()
print('College degree table columns:', list(college_degree_df.columns))
print('HS degree table columns:     ', list(hs_degree_df.columns))

Saved ../data/college_out_degree.csv  (5,168 rows)
Saved ../data/hs_in_degree.csv        (40,187 rows)

College degree table columns: ['college_id', 'year', 'n_hs_visited', 'col_name', 'col_city', 'col_st', 'col_zip', 'col_ctyname', 'col_ctyfips', 'col_type', 'col_inst_control', 'col_inst_size', 'col_endow_total', 'col_endow_per_fte', 'col_number_applied', 'col_number_admitted', 'col_number_enrolled_total', 'col_enrollment_rate', 'col_acceptance_rate', 'col_tuition_published', 'col_tuition_fees_ft']
HS degree table columns:      ['hs_id', 'year', 'n_colleges_visiting', 'hs_name', 'hs_city', 'hs_state', 'hs_zip', 'hs_ctyname', 'hs_cty_fips', 'hs_lat', 'hs_long', 'school_type', 'hs_school_level', 'hs_highest_grade_offered', 'hs_charter', 'hs_magnet', 'hs_title_i_status', 'hs_urban_centric_locale', 'hs_enrollment', 'hs_students_per_teacher', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more', 'hs_pct_free_or_reduced_price_